# 2 — Acesso e Inclusão

Objetivo deste notebook é realizar a ingestão, o recorte territorial para o município de Porto Alegre (código IBGE 4314902 / 431490) e o diagnóstico estrutural das bases tabulares e geoespaciais do projeto.

Fontes analisadas:

• Demografia e Renda (Censo 2022 - IBGE): Tabelas agregadas por setor censitário (demografia e renda do responsável; o bloco básico já vem embutido na malha de setores `.gpkg`).

• Saúde (CNES): Cadastro Nacional de Estabelecimentos de Saúde.

• Educação (INEP/Secretaria): Lista detalhada das escolas de Porto Alegre.

• Assistência Social: Camada oficial de equipamentos socioassistenciais do município.

• Território e Vulnerabilidade (IBGE): Malha de setores censitários do RS (`RS_setores_CD2022.gpkg`) e polígonos de Favelas e Comunidades Urbanas (FCUs).


In [1]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import pyogrio

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "1_raw"

print("Repositório:", ROOT)
print("Dados brutos:", RAW)


Repositório: /Users/aluno-81/Documents/projects/eld_dataset
Dados brutos: /Users/aluno-81/Documents/projects/eld_dataset/data/1_raw


In [2]:
def carregar_csv(nome, encoding, separador=";"):
    caminho = RAW / nome
    df = pd.read_csv(
        caminho,
        sep=separador,
        encoding=encoding,
        dtype=str,
    )
    return df, encoding, separador


In [3]:
demografia, enc_demografia, sep_demografia = carregar_csv(
    "Agregados_por_setores_demografia_BR.csv",
    encoding="utf-8",
)
renda, enc_renda, sep_renda = carregar_csv(
    "Agregados_por_setores_renda_responsavel_BR.csv",
    encoding="utf-8",
)
cnes, enc_cnes, sep_cnes = carregar_csv(
    "cnes_estabelecimentos.csv",
    encoding="cp1252",
)
caminho_escolas = next(
    (RAW / nome for nome in (
        "Análise - Tabela da lista das escolas - Detalhado.csv",
        "Tabela_da_lista_das_escolas_Detalhado.csv",
    ) if (RAW / nome).exists()),
    None,
)
if caminho_escolas is None:
    raise FileNotFoundError("Arquivo da lista detalhada de escolas não encontrado em data/1_raw")

escolas, enc_escolas, sep_escolas = carregar_csv(
    caminho_escolas.name,
    encoding="utf-8-sig",
    separador=",",
)

print("Bases tabulares carregadas:")
print("- Demografia IBGE:", demografia.shape, enc_demografia, sep_demografia)
print("- Renda IBGE:", renda.shape, enc_renda, sep_renda)
print("- CNES:", cnes.shape, enc_cnes, sep_cnes)
print("- Escolas:", escolas.shape, enc_escolas, sep_escolas)

# Conferência de acentuação e remoção de BOM
cnes_poa_tmp = cnes.loc[cnes["CO_IBGE"] == "431490"]
acentos_fantasia = int(cnes_poa_tmp["NO_FANTASIA"].dropna().str.contains(r"[À-ÿ]", regex=True).sum())
acentos_razao = int(cnes_poa_tmp["NO_RAZAO_SOCIAL"].dropna().str.contains(r"[À-ÿ]", regex=True).sum())

print("\nConferência de encoding / BOM:")
print(
    f"- CNES POA (NO_FANTASIA amostra): {cnes_poa_tmp['NO_FANTASIA'].dropna().head(3).tolist()} "
    f"| linhas com acento em NO_FANTASIA={acentos_fantasia}, NO_RAZAO_SOCIAL={acentos_razao} "
    "(DATASUS normaliza nomes para maiúsculas sem acento; cp1252 cobre bytes Latin-1 residuais do CSV nacional)"
)
print("- Escolas 1ª coluna (sem BOM):", repr(escolas.columns[0]))
print("- Escolas (amostra acentuada):", escolas["Restrição de Atendimento"].dropna().unique()[:3].tolist())
del cnes_poa_tmp


Bases tabulares carregadas:
- Demografia IBGE: (458772, 37) utf-8 ;
- Renda IBGE: (458772, 7) utf-8 ;
- CNES: (643594, 36) cp1252 ;
- Escolas: (988, 19) utf-8-sig ,

Conferência de encoding / BOM:
- CNES POA (NO_FANTASIA amostra): ['VANESSA ALECSSANDRA BRUNHERI PEREIRA', 'RENATA DUTRA DE OLIVEIRA', 'CLINICA GHENO'] | linhas com acento em NO_FANTASIA=0, NO_RAZAO_SOCIAL=0 (DATASUS normaliza nomes para maiúsculas sem acento; cp1252 cobre bytes Latin-1 residuais do CSV nacional)
- Escolas 1ª coluna (sem BOM): 'Restrição de Atendimento'
- Escolas (amostra acentuada): ['ESCOLA EM FUNCIONAMENTO E SEM RESTRIÇÃO DE ATENDIMENTO', 'ESCOLA PARALISADA', 'ESCOLA ATENDE EXCLUSIVAMENTE ALUNOS COM DEFICIÊNCIA']


In [4]:
setores = gpd.read_file(
    RAW / "RS_setores_CD2022.gpkg",
    layer="RS_setores_CD2022",
)

assistencia_camada = pyogrio.list_layers(RAW / "assistencia_social_poa.gpkg")[0][0]
assistencia = gpd.read_file(
    RAW / "assistencia_social_poa.gpkg",
    layer=assistencia_camada,
)

fcu = gpd.read_file(
    RAW / "poligonos_FCUs_shp" / "qg_2022_670_fcu_agreg.shp"
)

print("Bases espaciais carregadas:")
print("- Setores IBGE:", setores.shape, setores.crs)
print("- Assistência social:", assistencia.shape, assistencia.crs, "| camada:", assistencia_camada)
print("- FCUs:", fcu.shape, fcu.crs)

cpg = RAW / "poligonos_FCUs_shp" / "qg_2022_670_fcu_agreg.cpg"
print("- CPG do shapefile:", cpg.read_text(encoding="ascii").strip())


Bases espaciais carregadas:
- Setores IBGE: (25575, 37) EPSG:4674
- Assistência social: (35, 6) EPSG:4674 | camada: assistencia_social_poa_100_oficial
- FCUs: (12348, 8) EPSG:4674
- CPG do shapefile: UTF-8


In [5]:
# Recortes em memória. Nenhuma linha é removida da fonte carregada.
demografia_poa = demografia.loc[
    demografia["CD_setor"].astype("string").str.startswith("4314902", na=False)
]
renda_poa = renda.loc[
    renda["CD_SETOR"].astype("string").str.startswith("4314902", na=False)
]
cnes_poa = cnes.loc[cnes["CO_IBGE"].astype("string") == "431490"]

setores_poa = setores.loc[
    setores["CD_MUN"].astype("string") == "4314902"
    if "CD_MUN" in setores.columns
    else setores["CD_SETOR"].astype("string").str.startswith("4314902", na=False)
]
fcu_poa = fcu.loc[fcu["cd_mun"].astype("string") == "4314902"]

# A base de escolas já contém apenas Porto Alegre.
escolas_poa = escolas.loc[escolas["Município"].astype("string") == "Porto Alegre"]

# O GPKG de assistência já representa Porto Alegre.
assistencia_poa = assistencia.copy()

summary_dimensoes = pd.DataFrame([
    {"Base": "IBGE demografia CSV", "Antes": demografia.shape, "Depois": demografia_poa.shape, "Filtro": "CD_setor inicia 4314902"},
    {"Base": "IBGE renda CSV", "Antes": renda.shape, "Depois": renda_poa.shape, "Filtro": "CD_SETOR inicia 4314902"},
    {"Base": "CNES CSV", "Antes": cnes.shape, "Depois": cnes_poa.shape, "Filtro": "CO_IBGE = 431490"},
    {"Base": "Escolas CSV", "Antes": escolas.shape, "Depois": escolas_poa.shape, "Filtro": "Município = Porto Alegre"},
    {"Base": "Setores GPKG", "Antes": setores.shape, "Depois": setores_poa.shape, "Filtro": "município/setor de POA"},
    {"Base": "Assistência GPKG", "Antes": assistencia.shape, "Depois": assistencia_poa.shape, "Filtro": "base já limitada a POA"},
    {"Base": "FCUs SHP", "Antes": fcu.shape, "Depois": fcu_poa.shape, "Filtro": "cd_mun = 4314902"},
])

display(summary_dimensoes)
print(
    f"Setores GPKG POA: {len(setores_poa)} polígonos para "
    f"{setores_poa['CD_SETOR'].nunique()} setores únicos "
    f"({len(setores_poa) - setores_poa['CD_SETOR'].nunique()} partes extras de setores multipolígono -> usar dissolve na limpeza)."
)


,Base,Antes,Depois,Filtro
0,IBGE demografia CSV,"(458772, 37)","(2713, 37)",CD_setor inicia 4314902
1,IBGE renda CSV,"(458772, 7)","(2713, 7)",CD_SETOR inicia 4314902
2,CNES CSV,"(643594, 36)","(10294, 36)",CO_IBGE = 431490
3,Escolas CSV,"(988, 19)","(988, 19)",Município = Porto Alegre
4,Setores GPKG,"(25575, 37)","(2748, 37)",município/setor de POA
5,Assistência GPKG,"(35, 6)","(35, 6)",base já limitada a POA
6,FCUs SHP,"(12348, 8)","(125, 8)",cd_mun = 4314902


Setores GPKG POA: 2748 polígonos para 2744 setores únicos (4 partes extras de setores multipolígono -> usar dissolve na limpeza).


## Formato e organização das bases

In [6]:
formatos = pd.DataFrame([
    {"Arquivo": "Agregados_por_setores_demografia_BR.csv", "Tipo": "CSV", "Encoding": enc_demografia, "Separador": repr(sep_demografia), "CRS": "N/A"},
    {"Arquivo": "Agregados_por_setores_renda_responsavel_BR.csv", "Tipo": "CSV", "Encoding": enc_renda, "Separador": repr(sep_renda), "CRS": "N/A"},
    {"Arquivo": "cnes_estabelecimentos.csv", "Tipo": "CSV", "Encoding": enc_cnes, "Separador": repr(sep_cnes), "CRS": "N/A"},
    {"Arquivo": "Tabela_da_lista_das_escolas_Detalhado.csv", "Tipo": "CSV", "Encoding": enc_escolas, "Separador": repr(sep_escolas), "CRS": "N/A"},
    {"Arquivo": "RS_setores_CD2022.gpkg", "Tipo": "Geoespacial", "Encoding": "N/A", "Separador": "N/A", "CRS": str(setores.crs)},
    {"Arquivo": "assistencia_social_poa.gpkg", "Tipo": "Geoespacial", "Encoding": "N/A", "Separador": "N/A", "CRS": str(assistencia.crs)},
    {"Arquivo": "qg_2022_670_fcu_agreg.shp", "Tipo": "Shapefile", "Encoding": cpg.read_text(encoding="ascii").strip(), "Separador": "N/A", "CRS": str(fcu.crs)},
])

display(formatos)


,Arquivo,Tipo,Encoding,Separador,CRS
0,Agregados_por_setores_demografia_BR.csv,CSV,utf-8,';',N/A
1,Agregados_por_setores_renda_responsavel_BR.csv,CSV,utf-8,';',N/A
2,cnes_estabelecimentos.csv,CSV,cp1252,';',N/A
3,Tabela_da_lista_das_escolas_Detalhado.csv,CSV,utf-8-sig,"','",N/A
4,RS_setores_CD2022.gpkg,Geoespacial,N/A,N/A,EPSG:4674
5,assistencia_social_poa.gpkg,Geoespacial,N/A,N/A,EPSG:4674
6,qg_2022_670_fcu_agreg.shp,Shapefile,UTF-8,N/A,EPSG:4674


In [7]:
colunas_selecionadas = {
    "IBGE demografia": (
        demografia_poa,
        ["CD_setor", "V01006", "V01040", "V01041"],
    ),
    "IBGE renda": (
        renda_poa,
        ["CD_SETOR", "V06001", "V06004", "V06006"],
    ),
    "Setores GPKG": (
        setores_poa,
        ["CD_SETOR", "CD_MUN", "v0001", "AREA_KM2", "SITUACAO", "CD_SIT", "CD_TIPO", "CD_FCU", "NM_FCU", "geometry"],
    ),
    "CNES": (
        cnes_poa,
        ["CO_CNES", "CO_UNIDADE", "NO_FANTASIA", "NO_RAZAO_SOCIAL", "CO_IBGE", "TP_UNIDADE", "CO_NATUREZA_JUR", "DS_ESFERA_ADMINISTRATIVA", "CO_AMBULATORIAL_SUS", "CO_MOTIVO_DESAB", "NU_LATITUDE", "NU_LONGITUDE", "ST_ATEND_HOSPITALAR"],
    ),
    "Escolas": (
        escolas_poa,
        ["Município", "Restrição de Atendimento", "Categoria Administrativa", "Dependência Administrativa", "Latitude", "Longitude"],
    ),
    "Assistência GPKG": (
        assistencia_poa,
        ["categoria", "nome", "endereco", "latitude", "longitude", "geometry"],
    ),
    "FCUs SHP": (
        fcu_poa,
        ["cd_fcu", "nm_fcu", "cd_mun", "nm_mun", "geometry"],
    ),
}

# 1. Conferência de existência das colunas e dtypes nativos das bases espaciais
for nome, (df, colunas) in colunas_selecionadas.items():
    ausentes = [c for c in colunas if c not in df.columns]
    if ausentes:
        print(f"[AVISO] {nome} — colunas não encontradas:", ausentes)
    if isinstance(df, gpd.GeoDataFrame):
        presentes = [c for c in colunas if c in df.columns]
        print(f"\n== {nome} (dtypes nativos) ==")
        print(df[presentes].dtypes.to_string())

# 2. Validação de conversão numérica em cópias temporárias das colunas numéricas dos CSVs
# (mantém dtype=str na leitura para preservar zeros à esquerda em códigos, mas expõe vírgulas, "X" e espaços)
colunas_numericas_csv = {
    "IBGE demografia": (demografia_poa, ["V01006", "V01040", "V01041"]),
    "IBGE renda": (renda_poa, ["V06001", "V06004", "V06006"]),
    "CNES": (cnes_poa, ["NU_LATITUDE", "NU_LONGITUDE"]),
    "Escolas": (escolas_poa, ["Latitude", "Longitude"]),
}

resultados_numericos = []
for nome, (df, colunas) in colunas_numericas_csv.items():
    for col in colunas:
        serie_tmp = df[col].copy()
        nao_nulos = serie_tmp.notna()

        conv_bruta = pd.to_numeric(serie_tmp, errors="coerce")
        falhas_brutas = nao_nulos & conv_bruta.isna()

        serie_limpa = serie_tmp.str.strip().str.replace(",", ".", regex=False)
        conv_pos_virgula = pd.to_numeric(serie_limpa, errors="coerce")
        falhas_pos_virgula = nao_nulos & conv_pos_virgula.isna()

        exemplos_bloqueio = [repr(v) for v in serie_tmp.loc[falhas_brutas].unique()[:3]]

        resultados_numericos.append({
            "Base": nome,
            "Coluna": col,
            "Total": len(serie_tmp),
            "Nulos_originais": int((~nao_nulos).sum()),
            "Nao_convertem_bruto": int(falhas_brutas.sum()),
            "Nao_convertem_pos_virgula": int(falhas_pos_virgula.sum()),
            "Exemplos_bloqueio": exemplos_bloqueio,
        })

validacao_numerica = pd.DataFrame(resultados_numericos)
print("\n== Validação de tipagem numérica nos CSVs (cópias temporárias) ==")
display(validacao_numerica)

# 3. Indicadores categóricos do CNES (tratados como flags, não variáveis numéricas contínuas)
print("\n== CNES — distribuição dos indicadores de atendimento ==")
for col in ["ST_ATEND_HOSPITALAR", "CO_AMBULATORIAL_SUS"]:
    dist = cnes_poa[col].value_counts(dropna=False).to_dict()
    print(f"- {col}: {dist}")



== Setores GPKG (dtypes nativos) ==
CD_SETOR         str
CD_MUN           str
v0001          int64
AREA_KM2     float64
SITUACAO         str
CD_SIT           str
CD_TIPO          str
CD_FCU           str
NM_FCU           str
geometry    geometry

== Assistência GPKG (dtypes nativos) ==
categoria         str
nome              str
endereco          str
latitude      float64
longitude     float64
geometry     geometry

== FCUs SHP (dtypes nativos) ==
cd_fcu           str
nm_fcu           str
cd_mun           str
nm_mun           str
geometry    geometry

== Validação de tipagem numérica nos CSVs (cópias temporárias) ==


,Base,Coluna,Total,Nulos_originais,Nao_convertem_bruto,Nao_convertem_pos_virgula,Exemplos_bloqueio
0,IBGE demografia,V01006,2713,0,18,18,['X']
1,IBGE demografia,V01040,2713,0,31,31,['X']
2,IBGE demografia,V01041,2713,0,48,48,['X']
3,IBGE renda,V06001,2713,0,18,18,['X']
4,IBGE renda,V06004,2713,0,18,18,['X']
5,IBGE renda,V06006,2713,0,18,18,['X']
6,CNES,NU_LATITUDE,10294,92,0,0,[]
7,CNES,NU_LONGITUDE,10294,92,0,0,[]
8,Escolas,Latitude,988,0,153,153,[' ']
9,Escolas,Longitude,988,0,153,153,[' ']



== CNES — distribuição dos indicadores de atendimento ==
- ST_ATEND_HOSPITALAR: {'0.0': 10111, nan: 149, '1.0': 34}
- CO_AMBULATORIAL_SUS: {'NAO': 9953, 'SIM': 341}


## Auditoria dos setores censitários (multipolígonos no GPKG e redundância do CSV básico)


In [8]:
# 1. Auditoria dos 4 setores multipolígono em RS_setores_CD2022.gpkg (Porto Alegre)
duplicados_gpkg = setores_poa.loc[setores_poa.duplicated("CD_SETOR", keep=False)].copy()
duplicados_gpkg["area_parte_km2"] = duplicados_gpkg.to_crs(31982).geometry.area / 1e6
duplicados_gpkg = duplicados_gpkg.sort_values(["CD_SETOR", "area_parte_km2"], ascending=[True, False])

print("== Setores com múltiplas partes no GPKG de Porto Alegre ==")
display(
    duplicados_gpkg[["CD_SETOR", "NM_BAIRRO", "SITUACAO", "CD_SIT", "v0001", "AREA_KM2", "area_parte_km2"]]
)

setores_poa_dissolved = setores_poa.dissolve(by="CD_SETOR", aggfunc="first", as_index=False)
pop_bruta = int(setores_poa["v0001"].sum())
pop_unificada = int(setores_poa_dissolved["v0001"].sum())
print(
    f"Soma de v0001 antes do dissolve: {pop_bruta:,} | "
    f"após dissolve(aggfunc='first'): {pop_unificada:,} | "
    f"excesso duplicado evitado: +{pop_bruta - pop_unificada:,} moradores"
)

# 2. Auditoria de redundância: RS_setores_CD2022.gpkg vs. Agregados_por_setores_basico_BR.csv
caminho_basico = RAW / "Agregados_por_setores_basico_BR.csv"
if caminho_basico.exists():
    cols_audit = ["CD_SETOR", "CD_MUN", "v0001", "AREA_KM2", "SITUACAO", "CD_SIT", "CD_TIPO", "CD_FCU", "NM_FCU"]
    basico_audit = pd.read_csv(
        caminho_basico,
        sep=";",
        encoding="cp1252",
        dtype=str,
        usecols=cols_audit,
    )
    basico_poa_audit = basico_audit.loc[basico_audit["CD_MUN"] == "4314902"].set_index("CD_SETOR")
    gpkg_idx = setores_poa_dissolved.set_index("CD_SETOR")

    comuns = sorted(set(gpkg_idx.index) & set(basico_poa_audit.index))
    so_gpkg = len(set(gpkg_idx.index) - set(basico_poa_audit.index))
    so_csv = len(set(basico_poa_audit.index) - set(gpkg_idx.index))

    comparacao = []
    for col in ["CD_MUN", "SITUACAO", "CD_SIT", "CD_TIPO", "CD_FCU", "NM_FCU"]:
        diffs = int((gpkg_idx.loc[comuns, col].fillna("") != basico_poa_audit.loc[comuns, col].fillna("")).sum())
        comparacao.append({"Coluna": col, "Divergencias": diffs, "Detalhe": "comparação exata"})

    v0001_csv = pd.to_numeric(basico_poa_audit.loc[comuns, "v0001"])
    comparacao.append({
        "Coluna": "v0001",
        "Divergencias": int((gpkg_idx.loc[comuns, "v0001"] != v0001_csv).sum()),
        "Detalhe": f"soma CSV={int(v0001_csv.sum()):,} vs GPKG={pop_unificada:,}",
    })

    area_csv = pd.to_numeric(basico_poa_audit.loc[comuns, "AREA_KM2"].str.replace(",", ".", regex=False))
    max_diff_area = float((gpkg_idx.loc[comuns, "AREA_KM2"] - area_csv).abs().max())
    comparacao.append({
        "Coluna": "AREA_KM2",
        "Divergencias": int((gpkg_idx.loc[comuns, "AREA_KM2"] - area_csv).abs().gt(1e-6).sum()),
        "Detalhe": f"diff máx abs = {max_diff_area:.2e} (arredondamento decimal)",
    })

    print(
        f"\n== Comparação GPKG (dissolved) vs. CSV básico ({len(comuns)} setores comuns | "
        f"só GPKG={so_gpkg} | só CSV={so_csv}) =="
    )
    display(pd.DataFrame(comparacao))
    del basico_audit, basico_poa_audit
else:
    print("\n[INFO] Agregados_por_setores_basico_BR.csv não está presente em data/1_raw/ (auditoria já confirmada).")



== Setores com múltiplas partes no GPKG de Porto Alegre ==


,CD_SETOR,NM_BAIRRO,SITUACAO,CD_SIT,v0001,AREA_KM2,area_parte_km2
15582,431490205001131,Vila Conceição,Urbana,1,373,0.268018,0.238130
15583,431490205001131,Vila Conceição,Urbana,1,373,0.268018,0.029677
15690,431490205001259,Belém Novo,Urbana,2,544,5.245650,5.231462
15691,431490205001259,Belém Novo,Urbana,2,544,5.245650,0.010028
17288,431490205003016,Boa Vista do Sul,Rural,8,150,10.841593,10.616126
17287,431490205003016,Boa Vista do Sul,Rural,8,150,10.841593,0.216852
17302,431490205003029,Arquipélago,NaN,9,0,5.026277,4.676333
17301,431490205003029,Arquipélago,NaN,9,0,5.026277,0.346012


Soma de v0001 antes do dissolve: 1,333,912 | após dissolve(aggfunc='first'): 1,332,845 | excesso duplicado evitado: +1,067 moradores



== Comparação GPKG (dissolved) vs. CSV básico (2744 setores comuns | só GPKG=0 | só CSV=0) ==


,Coluna,Divergencias,Detalhe
0,CD_MUN,0,comparação exata
1,SITUACAO,0,comparação exata
2,CD_SIT,0,comparação exata
3,CD_TIPO,0,comparação exata
4,CD_FCU,0,comparação exata
5,NM_FCU,0,comparação exata
6,v0001,0,"soma CSV=1,332,845 vs GPKG=1,332,845"
7,AREA_KM2,0,diff máx abs = 5.00e-08 (arredondamento decimal)


In [9]:
# 1. Auditoria de CD_FCU e NM_FCU em setores_poa_dissolved vs. fcu_poa
cd_fcu_setor = setores_poa_dissolved["CD_FCU"]
nm_fcu_setor = setores_poa_dissolved["NM_FCU"]

resumo_cd_fcu = pd.DataFrame([
    {"Categoria": "Valor '.' (sem FCU)", "Setores": int((cd_fcu_setor == ".").sum())},
    {"Categoria": "Nulo (NaN)", "Setores": int(cd_fcu_setor.isna().sum())},
    {"Categoria": "String vazia", "Setores": int((cd_fcu_setor.fillna("") == "").sum() - cd_fcu_setor.isna().sum())},
    {"Categoria": "Código FCU válido", "Setores": int((cd_fcu_setor.notna() & ~cd_fcu_setor.isin([".", ""])).sum())},
])
print("== Preenchimento de CD_FCU em setores_poa (após dissolve) ==")
display(resumo_cd_fcu)
print("NM_FCU — não nulos:", int(nm_fcu_setor.notna().sum()), "| nulos:", int(nm_fcu_setor.isna().sum()))

codigos_setor = set(cd_fcu_setor.loc[cd_fcu_setor.notna() & ~cd_fcu_setor.isin([".", ""])].astype(str))
codigos_shp = set(fcu_poa["cd_fcu"].dropna().astype(str))

so_no_shp = sorted(codigos_shp - codigos_setor)
so_nos_setores = sorted(codigos_setor - codigos_shp)

print(f"\n== Comparação de códigos FCU (setores vs. shapefile fcu_poa) ==")
print(f"- fcu_poa: {len(fcu_poa)} feições | {fcu_poa['cd_fcu'].nunique()} cd_fcu únicos")
print(f"- setores_poa: {len(codigos_setor)} CD_FCU únicos válidos (em {int((cd_fcu_setor.notna() & ~cd_fcu_setor.isin(['.', ''])).sum())} setores)")
print(f"- Códigos em comum: {len(codigos_setor & codigos_shp)}")
print(f"- Só em fcu_poa (ausentes em CD_FCU dos setores): {len(so_no_shp)}")
print(f"- Só em setores_poa (ausentes em fcu_poa): {len(so_nos_setores)}")

if so_no_shp:
    print("\nFCUs presentes só no shapefile fcu_poa:")
    display(fcu_poa.loc[fcu_poa["cd_fcu"].astype(str).isin(so_no_shp), ["cd_fcu", "nm_fcu"]].head(15))

# Conferência cruzada entre CD_TIPO == '1' e presença de CD_FCU válido
tem_cd_fcu = cd_fcu_setor.notna() & ~cd_fcu_setor.isin([".", ""])
print("\n== Cruzamento CD_TIPO == '1' vs. CD_FCU válido nos setores ==")
display(pd.crosstab(setores_poa_dissolved["CD_TIPO"] == "1", tem_cd_fcu, rownames=["CD_TIPO == '1'"], colnames=["CD_FCU_valido"]))


== Preenchimento de CD_FCU em setores_poa (após dissolve) ==


,Categoria,Setores
0,Valor '.' (sem FCU),2421
1,Nulo (NaN),0
2,String vazia,0
3,Código FCU válido,323


NM_FCU — não nulos: 323 | nulos: 2421

== Comparação de códigos FCU (setores vs. shapefile fcu_poa) ==
- fcu_poa: 125 feições | 125 cd_fcu únicos
- setores_poa: 125 CD_FCU únicos válidos (em 323 setores)
- Códigos em comum: 125
- Só em fcu_poa (ausentes em CD_FCU dos setores): 0
- Só em setores_poa (ausentes em fcu_poa): 0

== Cruzamento CD_TIPO == '1' vs. CD_FCU válido nos setores ==


CD_FCU_valido,False,True
CD_TIPO == '1',,
False,2421,0
True,0,323


In [10]:
# 2. Consistência populacional: v0001 (setores GPKG dissolved) vs. V01006 (demografia CSV)
pop_check = setores_poa_dissolved[["CD_SETOR", "NM_BAIRRO", "SITUACAO", "CD_SIT", "CD_TIPO", "v0001"]].merge(
    demografia_poa[["CD_setor", "V01006"]],
    left_on="CD_SETOR",
    right_on="CD_setor",
    how="left",
)

sem_demo = pop_check.loc[pop_check["CD_setor"].isna()]
com_sigilo = pop_check.loc[pop_check["V01006"] == "X"]
comparaveis = pop_check.loc[pop_check["CD_setor"].notna() & (pop_check["V01006"] != "X")].copy()
comparaveis["V01006_num"] = pd.to_numeric(comparaveis["V01006"], errors="coerce")
comparaveis["diff"] = comparaveis["v0001"] - comparaveis["V01006_num"]
divergentes = comparaveis.loc[comparaveis["diff"] != 0]

resumo_pop = pd.DataFrame([
    {"Grupo": "Total de setores (GPKG dissolved)", "Setores": len(pop_check), "Soma_v0001": int(pop_check["v0001"].sum())},
    {"Grupo": "Sem linha em demografia_poa", "Setores": len(sem_demo), "Soma_v0001": int(sem_demo["v0001"].sum())},
    {"Grupo": "Com sigilo ('X') em V01006", "Setores": len(com_sigilo), "Soma_v0001": int(com_sigilo["v0001"].sum())},
    {"Grupo": "Comparáveis (numéricos)", "Setores": len(comparaveis), "Soma_v0001": int(comparaveis["v0001"].sum())},
    {"Grupo": "Divergentes (v0001 != V01006)", "Setores": len(divergentes), "Soma_v0001": int(divergentes["v0001"].sum())},
])

print("== Consistência v0001 (GPKG) vs. V01006 (Demografia) ==")
display(resumo_pop)
print(
    f"Nos {len(comparaveis)} setores comparáveis: "
    f"soma v0001 = {int(comparaveis['v0001'].sum()):,} | "
    f"soma V01006 = {int(comparaveis['V01006_num'].sum()):,} | "
    f"divergências = {len(divergentes)}"
)

if len(divergentes) > 0:
    print("\nSetores com divergência entre v0001 e V01006 (top 15 por |diff|):")
    display(
        divergentes.assign(abs_diff=divergentes["diff"].abs())
        .sort_values("abs_diff", ascending=False)
        [["CD_SETOR", "NM_BAIRRO", "CD_SIT", "CD_TIPO", "v0001", "V01006_num", "diff"]]
        .head(15)
    )


== Consistência v0001 (GPKG) vs. V01006 (Demografia) ==


,Grupo,Setores,Soma_v0001
0,Total de setores (GPKG dissolved),2744,1332845
1,Sem linha em demografia_poa,31,0
2,Com sigilo ('X') em V01006,18,2791
3,Comparáveis (numéricos),2695,1330054
4,Divergentes (v0001 != V01006),0,0


Nos 2695 setores comparáveis: soma v0001 = 1,330,054 | soma V01006 = 1,330,054 | divergências = 0


In [11]:
# 3. Distribuição de setores e população (v0001) por CD_SIT e CD_TIPO
pop_total_poa = setores_poa_dissolved["v0001"].sum()

dist_sit = (
    setores_poa_dissolved
    .assign(SITUACAO=setores_poa_dissolved["SITUACAO"].fillna("(Massa d'água / NaN)"))
    .groupby(["CD_SIT", "SITUACAO"], dropna=False)
    .agg(
        setores=("CD_SETOR", "count"),
        pop_v0001=("v0001", "sum"),
        area_km2=("AREA_KM2", "sum"),
    )
    .reset_index()
)
dist_sit["pct_pop"] = (dist_sit["pop_v0001"] / pop_total_poa * 100).round(2)

print("== Distribuição por situação do setor (CD_SIT / SITUACAO) ==")
display(dist_sit)

dist_tipo = (
    setores_poa_dissolved
    .groupby("CD_TIPO", dropna=False)
    .agg(
        setores=("CD_SETOR", "count"),
        setores_pop_zero=("v0001", lambda s: int((s == 0).sum())),
        pop_v0001=("v0001", "sum"),
        area_km2=("AREA_KM2", "sum"),
        bairros_exemplo=("NM_BAIRRO", lambda s: ", ".join(s.dropna().unique()[:3])),
    )
    .reset_index()
)
dist_tipo["pct_pop"] = (dist_tipo["pop_v0001"] / pop_total_poa * 100).round(2)

print("\n== Distribuição por tipo de setor (CD_TIPO) ==")
display(dist_tipo)

print("\n== Detalhe dos setores com CD_TIPO em {'2', '6', '7'} ==")
display(
    setores_poa_dissolved.loc[
        setores_poa_dissolved["CD_TIPO"].isin(["2", "6", "7"]),
        ["CD_SETOR", "CD_TIPO", "CD_SIT", "SITUACAO", "NM_BAIRRO", "v0001", "AREA_KM2"],
    ].sort_values(["CD_TIPO", "v0001"], ascending=[True, False])
)


== Distribuição por situação do setor (CD_SIT / SITUACAO) ==


,CD_SIT,SITUACAO,setores,pop_v0001,area_km2,pct_pop
0,1,Urbana,2630,1293352,233.246227,97.04
1,2,Urbana,87,37076,192.244718,2.78
2,5,Rural,2,277,0.111510,0.02
3,7,Rural,8,329,1.320700,0.02
4,8,Rural,11,1811,50.386596,0.14
5,9,(Massa d'água / NaN),6,0,18.080414,0.00



== Distribuição por tipo de setor (CD_TIPO) ==


,CD_TIPO,setores,setores_pop_zero,pop_v0001,area_km2,bairros_exemplo,pct_pop
0,0,2359,7,1152474,433.364946,"Centro Histórico, Praia de Belas, Menino-Deus",86.47
1,1,323,1,175519,23.221584,"Bom Jesus, Jardim Carvalho, Morro Santana",13.17
2,2,1,1,0,0.267925,Partenon,0.00
3,4,34,17,231,36.615937,"Cristal, Nonoai, Teresópolis",0.02
4,5,8,2,302,1.334435,"Extrema, Sétimo Céu, Boa Vista do Sul",0.02
5,6,7,3,2801,0.219834,"Coronel Aparício Borges, Partenon, Santa Tereza",0.21
6,7,6,0,398,0.178204,"Santa Tereza, Nonoai, Ipanema",0.03
7,9,6,0,1120,0.187299,"Glória, Três Figueiras, Azenha",0.08



== Detalhe dos setores com CD_TIPO em {'2', '6', '7'} ==


,CD_SETOR,CD_TIPO,CD_SIT,SITUACAO,NM_BAIRRO,v0001,AREA_KM2
2504,431490205002793,2,1,Urbana,Partenon,0,0.267925
428,431490205000465,6,1,Urbana,Coronel Aparício Borges,2033,0.076790
491,431490205000534,6,1,Urbana,Partenon,474,0.074846
2619,431490205002908,6,1,Urbana,Partenon,247,0.030542
644,431490205000701,6,1,Urbana,Santa Tereza,47,0.008470
772,431490205000843,6,1,Urbana,Teresópolis,0,0.010352
2621,431490205002910,6,1,Urbana,Partenon,0,0.012257
2738,431490205003027,6,2,Urbana,Agronomia,0,0.006577
755,431490205000825,7,1,Urbana,Nonoai,119,0.016763
643,431490205000700,7,1,Urbana,Santa Tereza,93,0.013217


In [12]:
# 4. Inspeção dos 18 setores com sigilo (V01006 == 'X') e dos 6 setores de CD_TIPO == '7'
# (operação somente leitura em cópia temporária)
auditoria_ibge = (
    setores_poa_dissolved[["CD_SETOR", "CD_TIPO", "CD_SIT", "SITUACAO", "NM_BAIRRO", "v0001", "AREA_KM2"]]
    .merge(
        demografia_poa[["CD_setor", "V01006", "V01040", "V01041"]],
        left_on="CD_SETOR",
        right_on="CD_setor",
        how="left",
    )
    .merge(
        renda_poa[["CD_SETOR", "V06006"]],
        on="CD_SETOR",
        how="left",
    )
)

auditoria_ibge["densidade_hab_km2"] = (auditoria_ibge["v0001"] / auditoria_ibge["AREA_KM2"]).round(1)
v40 = pd.to_numeric(auditoria_ibge["V01040"], errors="coerce")  # 'X' -> NaN (nunca zero)
v41 = pd.to_numeric(auditoria_ibge["V01041"], errors="coerce")  # 'X' -> NaN (nunca zero)
auditoria_ibge["idosos_60_mais"] = v40 + v41  # preserva NaN se qualquer faixa estiver em sigilo
pop_valida = auditoria_ibge["v0001"].where(auditoria_ibge["v0001"] > 0)
auditoria_ibge["pct_idosos"] = (auditoria_ibge["idosos_60_mais"] / pop_valida * 100).round(2)

# --- Tabela 1: 18 setores com sigilo em V01006 ---
sigilo_18 = auditoria_ibge.loc[auditoria_ibge["V01006"] == "X"].sort_values(["CD_TIPO", "v0001"], ascending=[True, False])
print(f"== 18 setores com sigilo (V01006 == 'X') | soma v0001 = {int(sigilo_18['v0001'].sum()):,} ==")
display(
    sigilo_18[["CD_SETOR", "CD_TIPO", "CD_SIT", "NM_BAIRRO", "v0001", "AREA_KM2", "densidade_hab_km2", "V01040", "V01041", "V06006"]]
)

resumo_sigilo_tipo = (
    sigilo_18.groupby("CD_TIPO")
    .agg(setores=("CD_SETOR", "count"), pop_v0001=("v0001", "sum"))
    .reset_index()
)
resumo_sigilo_tipo["eliminado_regra_tipo6"] = resumo_sigilo_tipo["CD_TIPO"] == "6"
print("\nResumo dos 18 setores com V01006 == 'X' por CD_TIPO:")
display(resumo_sigilo_tipo)

# --- Tabela 2: 6 setores de CD_TIPO == '7' ---
tipo7 = auditoria_ibge.loc[auditoria_ibge["CD_TIPO"] == "7"].sort_values("densidade_hab_km2", ascending=False)
print(f"\n== 6 setores de CD_TIPO == '7' | soma v0001 = {int(tipo7['v0001'].sum()):,} ==")
display(
    tipo7[["CD_SETOR", "CD_TIPO", "CD_SIT", "NM_BAIRRO", "v0001", "AREA_KM2", "densidade_hab_km2", "V01006", "V01040", "V01041", "pct_idosos", "V06006"]]
)

# Referência de distribuição (percentis) nos setores mantidos de POA (sem CD_SIT=9, CD_TIPO em 2/6)
base_ref = auditoria_ibge.loc[
    (auditoria_ibge["CD_SIT"] != "9") & (~auditoria_ibge["CD_TIPO"].isin(["2", "6"])) & (auditoria_ibge["v0001"] > 0)
]
print("\nReferência nos setores habitados mantidos de POA:")
print(
    "Densidade (hab/km²) — mediana:", round(base_ref["densidade_hab_km2"].median(), 1),
    "| p95:", round(base_ref["densidade_hab_km2"].quantile(0.95), 1),
    "| p99:", round(base_ref["densidade_hab_km2"].quantile(0.99), 1),
    "| máx:", round(base_ref["densidade_hab_km2"].max(), 1),
)
print(
    "% idosos (60+) — mediana:", round(base_ref["pct_idosos"].median(), 2),
    "| p95:", round(base_ref["pct_idosos"].quantile(0.95), 2),
    "| p99:", round(base_ref["pct_idosos"].quantile(0.99), 2),
    "| máx:", round(base_ref["pct_idosos"].max(), 2),
)


== 18 setores com sigilo (V01006 == 'X') | soma v0001 = 2,791 ==


,CD_SETOR,CD_TIPO,CD_SIT,NM_BAIRRO,v0001,AREA_KM2,densidade_hab_km2,V01040,V01041,V06006
1927,431490205002143,0,1,Sarandi,4,0.079324,50.4,X,X,X
2734,431490205003023,4,2,Arquipélago,10,0.370901,27.0,X,X,X
2736,431490205003025,4,2,Lomba do Pinheiro,10,0.493386,20.3,X,X,X
2705,431490205002994,4,2,Arquipélago,7,2.052185,3.4,X,X,X
2414,431490205002694,4,1,Praia de Belas,6,1.213909,4.9,X,X,X
2626,431490205002915,4,1,Santa Rosa de Lima,5,0.020820,240.2,X,X,X
2739,431490205003028,4,2,Arquipélago,3,2.619018,1.1,X,X,X
759,431490205000830,4,2,Nonoai,2,0.266202,7.5,X,X,X
2545,431490205002834,5,7,Boa Vista do Sul,19,0.008598,2209.9,X,X,X
428,431490205000465,6,1,Coronel Aparício Borges,2033,0.076790,26474.9,X,X,X



Resumo dos 18 setores com V01006 == 'X' por CD_TIPO:


,CD_TIPO,setores,pop_v0001,eliminado_regra_tipo6
0,0,1,4,False
1,4,7,43,False
2,5,1,19,False
3,6,3,2327,True
4,7,6,398,False



== 6 setores de CD_TIPO == '7' | soma v0001 = 398 ==


,CD_SETOR,CD_TIPO,CD_SIT,NM_BAIRRO,v0001,AREA_KM2,densidade_hab_km2,V01006,V01040,V01041,pct_idosos,V06006
2683,431490205002972,7,1,Cristal,57,0.001596,35721.4,X,X,X,NaN,X
2609,431490205002898,7,1,Mário Quintana,61,0.007271,8389.1,X,X,X,NaN,X
755,431490205000825,7,1,Nonoai,119,0.016763,7099.0,X,X,X,NaN,X
643,431490205000700,7,1,Santa Tereza,93,0.013217,7036.6,X,X,X,NaN,X
1130,431490205001258,7,1,Belém Novo,65,0.108442,599.4,X,X,X,NaN,X
1030,431490205001142,7,1,Ipanema,3,0.030916,97.0,X,X,X,NaN,X



Referência nos setores habitados mantidos de POA:
Densidade (hab/km²) — mediana: 9471.4 | p95: 39483.7 | p99: 75190.3 | máx: 491771.7
% idosos (60+) — mediana: 24.36 | p95: 37.76 | p99: 43.89 | máx: 51.95


## Passo 3 — Auditoria fechada das bases de pontos

Esta é a seção única e consolidada da auditoria: todos os testes abaixo são somente leitura, não removem registros e não salvam bases intermediárias.

In [13]:
# Passo 3 — auditoria fechada e somente leitura

# Fronteira de POA pela malha dissolvida por setor
setores_poa_dissolved = setores_poa.dissolve(by="CD_SETOR", aggfunc="first", as_index=False)
poa_boundary = setores_poa_dissolved.union_all()

# ============================
# CNES
# ============================
cnes_ativos = cnes_poa.loc[cnes_poa["CO_MOTIVO_DESAB"].isna()].copy()
tipos_selecionados = {"1", "2", "4", "5", "7", "20", "21", "70", "73"}
cnes_tipos = cnes_ativos.loc[cnes_ativos["TP_UNIDADE"].astype("string").isin(tipos_selecionados)].copy()
natureza_publica = (
    cnes_tipos["CO_NATUREZA_JUR"].astype("string").str.startswith("1", na=False)
    | cnes_tipos["CO_NATUREZA_JUR"].astype("string").eq("2011")
)
cnes_final = cnes_tipos.loc[
    cnes_tipos["CO_AMBULATORIAL_SUS"].astype("string").eq("SIM") | natureza_publica
].copy()

cnes_lat = pd.to_numeric(cnes_final["NU_LATITUDE"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
cnes_lon = pd.to_numeric(cnes_final["NU_LONGITUDE"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
cnes_coord_ok = cnes_lat.notna() & cnes_lon.notna()

cnes_points = gpd.GeoSeries(
    gpd.points_from_xy(cnes_lon, cnes_lat, crs="EPSG:4674"),
    index=cnes_final.index,
)
cnes_inside = cnes_points.within(poa_boundary)

print("== CNES — funil completo ==")
print("10.294 total POA ->", len(cnes_poa), "no recorte POA")
print("->", len(cnes_ativos), "ativos")
print("->", len(cnes_tipos), "nos tipos selecionados")
print("->", len(cnes_final), "no recorte SUS/público")

print("\n== CNES — teste espacial corrigido ==")
print("Unidades no recorte final:", len(cnes_final))
print("Sem coordenada válida:", int((~cnes_coord_ok).sum()))
print("Fora do contorno de POA:", int((cnes_coord_ok & ~cnes_inside).sum()))

cnes_capsi_leste = cnes_final.loc[
    cnes_final["NO_FANTASIA"].astype("string").str.strip().str.upper().eq("CAPSI III LESTE")
][["CO_CNES", "NO_FANTASIA", "TP_UNIDADE", "NU_LATITUDE", "NU_LONGITUDE"]].copy()
print("\n== CNES — CAPSI III LESTE ==")
display(cnes_capsi_leste)

# 17 exclusões explícitas decididas na auditoria
cnes_exclusoes_ids = {
    "8604428", "7355106", "7355122", "7533446", "7355114", "2237156",
    "7989954", "8083649", "915742", "4758749", "6957560", "9948007",
    "6957544", "4560191", "804959", "9827153", "9736271",
}
cnes_exclusoes = cnes_final.loc[
    cnes_final["CO_CNES"].astype("string").isin(cnes_exclusoes_ids)
][["CO_CNES", "NO_FANTASIA", "TP_UNIDADE"]].copy().sort_values("NO_FANTASIA")
print("== CNES — 17 unidades a excluir ==")
print("Quantidade:", len(cnes_exclusoes))
display(cnes_exclusoes)

cnes_limpo_auditoria = cnes_final.loc[
    ~cnes_final["CO_CNES"].astype("string").isin(cnes_exclusoes_ids)
].copy()
print("== CNES — recorte pronto para limpeza ==")
print("Total após 17 exclusões:", len(cnes_limpo_auditoria))

hospitais_mantidos = cnes_limpo_auditoria.loc[
    cnes_limpo_auditoria["TP_UNIDADE"].astype("string").isin({"5", "7"})
].copy()
hospitais_publicos = hospitais_mantidos.loc[
    hospitais_mantidos["CO_NATUREZA_JUR"].astype("string").str.startswith("1", na=False)
    | hospitais_mantidos["CO_NATUREZA_JUR"].astype("string").eq("2011")
].copy()
hospitais_privados_conveniados = hospitais_mantidos.loc[
    hospitais_mantidos["CO_NATUREZA_JUR"].astype("string").str.startswith("3", na=False)
].copy()

print("== CNES — 21 hospitais mantidos: 8 públicos + 13 privados/filantrópicos ==")
print("Hospitais públicos:", len(hospitais_publicos))
display(hospitais_publicos[["CO_CNES", "NO_FANTASIA", "TP_UNIDADE", "CO_NATUREZA_JUR", "CO_AMBULATORIAL_SUS"]].sort_values("NO_FANTASIA"))
print("Hospitais privados/filantrópicos conveniados ao SUS:", len(hospitais_privados_conveniados))
display(hospitais_privados_conveniados[["CO_CNES", "NO_FANTASIA", "TP_UNIDADE", "CO_NATUREZA_JUR", "CO_AMBULATORIAL_SUS"]].sort_values("NO_FANTASIA"))

# ============================
# Escolas
# ============================
# A única regra ativa exclui paralisadas; educação especial permanece.
escolas_ativas = escolas_poa.loc[
    ~escolas_poa["Restrição de Atendimento"].astype("string").str.contains("PARALISADA", case=False, na=False)
].copy()

escola_lat = pd.to_numeric(escolas_ativas["Latitude"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
escola_lon = pd.to_numeric(escolas_ativas["Longitude"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
escola_coord_ok = escola_lat.notna() & escola_lon.notna()

cat = escolas_ativas["Categoria Administrativa"].astype("string").str.strip()
conv = escolas_ativas["Conveniada Poder Público"].astype("string").str.strip()
publica = cat.eq("Pública")
privada_conveniada = cat.eq("Privada") & conv.eq("Sim")
privada_particular = cat.eq("Privada") & ~conv.eq("Sim")

cross_escolas = pd.DataFrame([
    {
        "Grupo": "Pública",
        "Coordenada válida": int((publica & escola_coord_ok).sum()),
        "Coordenada em branco": int((publica & ~escola_coord_ok).sum()),
        "Total": int(publica.sum()),
    },
    {
        "Grupo": "Privada Conveniada",
        "Coordenada válida": int((privada_conveniada & escola_coord_ok).sum()),
        "Coordenada em branco": int((privada_conveniada & ~escola_coord_ok).sum()),
        "Total": int(privada_conveniada.sum()),
    },
    {
        "Grupo": "Privada Particular",
        "Coordenada válida": int((privada_particular & escola_coord_ok).sum()),
        "Coordenada em branco": int((privada_particular & ~escola_coord_ok).sum()),
        "Total": int(privada_particular.sum()),
    },
])
print("== Escolas — filtro ativo (~PARALISADA) ==")
print("Total não paralisadas:", len(escolas_ativas))
display(cross_escolas)

escolas_especiais = escolas_ativas.loc[
    escolas_ativas["Restrição de Atendimento"].astype("string").str.contains("DEFICIÊNCIA", case=False, na=False)
]
print("Educação especial:", len(escolas_especiais))
display(escolas_especiais[[
    "Escola", "Categoria Administrativa", "Conveniada Poder Público", "Endereço",
    "Etapas e Modalidade de Ensino Oferecidas", "Latitude", "Longitude",
]])

# ============================
# Registros explícitos solicitados
# ============================
emeff_sao_pedro = escolas_poa.loc[
    escolas_poa["Escola"].astype("string").str.strip().str.upper() == "EMEF SAO PEDRO"
][["Escola", "Endereço", "Categoria Administrativa", "Restrição de Atendimento",
   "Etapas e Modalidade de Ensino Oferecidas", "Latitude", "Longitude"]]
emeff_lat = pd.to_numeric(emeff_sao_pedro["Latitude"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
emeff_lon = pd.to_numeric(emeff_sao_pedro["Longitude"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
emeff_point_31982 = gpd.GeoSeries(
    gpd.points_from_xy(emeff_lon, emeff_lat, crs="EPSG:4674"),
).to_crs("EPSG:31982")
poa_boundary_31982 = gpd.GeoSeries([poa_boundary], crs="EPSG:4674").to_crs("EPSG:31982")
distancia_emeff_m = float(emeff_point_31982.distance(poa_boundary_31982.iloc[0].boundary).iloc[0])
print("== EMEF SAO PEDRO — mantida; distância ao limite ==")
print(f"Distância calculada em EPSG:31982: {distancia_emeff_m:.2f} m")
display(emeff_sao_pedro)

print("== Assistência Social — diagnóstico espacial ==")
print(assistencia_poa["categoria"].value_counts().sort_index().to_string())
assist_points = gpd.GeoSeries(assistencia_poa.geometry, crs=assistencia_poa.crs)
assist_inside = assist_points.to_crs("EPSG:4674").within(poa_boundary)
print("Pontos fora do contorno de POA:", int((~assist_inside).sum()))

assist_part = assistencia_poa.loc[
    assistencia_poa["nome"].astype("string").str.contains("Partenon", case=False, na=False)
][["categoria", "nome", "endereco", "latitude", "longitude"]]
print("== CRAS/CREAS Partenon — coordenada compartilhada ==")
display(assist_part.sort_values("categoria"))

== CNES — funil completo ==


10.294 total POA -> 10294 no recorte POA
-> 9203 ativos
-> 225 nos tipos selecionados
-> 200 no recorte SUS/público

== CNES — teste espacial corrigido ==
Unidades no recorte final: 200
Sem coordenada válida: 0
Fora do contorno de POA: 1

== CNES — CAPSI III LESTE ==


,CO_CNES,NO_FANTASIA,TP_UNIDADE,NU_LATITUDE,NU_LONGITUDE
563059,8604428,CAPSI III LESTE,70,-30.0619309,-50.1557879


== CNES — 17 unidades a excluir ==
Quantidade: 17


,CO_CNES,NO_FANTASIA,TP_UNIDADE
563059,8604428,CAPSI III LESTE,70
465336,7355106,CASE II,2
465338,7355122,CASEF,2
478107,7533446,CONSULTORIO NA RUA GHC,2
465337,7355114,CSE COMUNIDADE SOCIO EDUCATIVA,2
85136,2237156,HBMPA,5
512730,7989954,HOSPITAL GERAL DE PORTO ALEGRE,5
520080,8083649,POLICLINICA,4
49573,915742,POLICLINICA MILITAR DE PORTO ALEGRE,4
273292,4758749,PRONTO ATENDIMENTO DE CAMPANHA BOM JESUS,73


== CNES — recorte pronto para limpeza ==
Total após 17 exclusões: 183
== CNES — 21 hospitais mantidos: 8 públicos + 13 privados/filantrópicos ==
Hospitais públicos: 8


... (7 outputs left out)

== EMEF SAO PEDRO — mantida; distância ao limite ==
Distância calculada em EPSG:31982: 1.48 m


,Escola,Endereço,Categoria Administrativa,Restrição de Atendimento,Etapas e Modalidade de Ensino Oferecidas,Latitude,Longitude
464,EMEF SAO PEDRO,"AV DEPUTADO ADAO PRETTO, 1190 LOMBA DO PINHEIR...",Pública,ESCOLA EM FUNCIONAMENTO E SEM RESTRIÇÃO DE ATE...,"Educação Infantil, Ensino Fundamental, Educaçã...",-30.1128568,-51.1008335


== Assistência Social — diagnóstico espacial ==
categoria
CRAS          23
CREAS          9
Centro POP     3
Pontos fora do contorno de POA: 0
== CRAS/CREAS Partenon — coordenada compartilhada ==


,categoria,nome,endereco,latitude,longitude
28,CRAS,CRAS Partenon,"Rua Barão do Amazonas, 1957, Partenon, Porto A...",-30.060419,-51.192499
9,CREAS,CREAS Partenon,"Rua Barão do Amazonas, 1957, Partenon, Porto A...",-30.060419,-51.192499


## Passo 4 — Limpeza, padronização e exportação

Esta seção aplica as regras fechadas pela auditoria e escreve os resultados finais em `data/3_done/`. Os arquivos intermediários não serão salvos separadamente.

In [14]:
# Passo 4 — limpeza, padronização e exportação em EPSG:4674
from pathlib import Path

DONE = ROOT / "data" / "3_done"
DONE.mkdir(parents=True, exist_ok=True)

# ---------------- Setores censitários ----------------
setores_poa_limpo = setores_poa.dissolve(
    by="CD_SETOR", aggfunc="first", as_index=False
).copy()

excluir_setores = (
    setores_poa_limpo["CD_TIPO"].astype("string").eq("6")
    | setores_poa_limpo["CD_SIT"].astype("string").eq("9")
    | setores_poa_limpo["CD_TIPO"].astype("string").eq("2")
)
setores_poa_limpo = setores_poa_limpo.loc[~excluir_setores].copy()

setores_poa_limpo = setores_poa_limpo.merge(
    demografia_poa[["CD_setor", "V01006", "V01040", "V01041"]],
    left_on="CD_SETOR",
    right_on="CD_setor",
    how="left",
)
setores_poa_limpo = setores_poa_limpo.merge(
    renda_poa[["CD_SETOR", "V06001", "V06004", "V06006"]],
    on="CD_SETOR",
    how="left",
)
if "CD_setor" in setores_poa_limpo.columns:
    setores_poa_limpo = setores_poa_limpo.drop(columns=["CD_setor"])

for col in ["V01006", "V01040", "V01041", "V06001", "V06004", "V06006"]:
    setores_poa_limpo[col] = pd.to_numeric(setores_poa_limpo[col], errors="coerce")

setores_poa_limpo["AREA_KM2"] = pd.to_numeric(setores_poa_limpo["AREA_KM2"], errors="coerce")
setores_poa_limpo["v0001"] = pd.to_numeric(setores_poa_limpo["v0001"], errors="coerce")
setores_poa_limpo["densidade_hab_km2"] = (
    setores_poa_limpo["v0001"] / setores_poa_limpo["AREA_KM2"]
).replace([float("inf"), float("-inf")], pd.NA)
setores_poa_limpo.loc[setores_poa_limpo["v0001"] == 0, "densidade_hab_km2"] = 0

setores_poa_limpo["idosos_60_mais"] = (
    setores_poa_limpo["V01040"] + setores_poa_limpo["V01041"]
)
setores_poa_limpo["pct_idosos"] = float("nan")
setores_poa_limpo.loc[setores_poa_limpo["v0001"] > 0, "pct_idosos"] = (
    100
    * setores_poa_limpo.loc[setores_poa_limpo["v0001"] > 0, "idosos_60_mais"]
    / setores_poa_limpo.loc[setores_poa_limpo["v0001"] > 0, "v0001"]
)
setores_poa_limpo["pct_idosos"] = pd.to_numeric(setores_poa_limpo["pct_idosos"], errors="coerce")
setores_poa_limpo["idosos_60_mais"] = pd.to_numeric(setores_poa_limpo["idosos_60_mais"], errors="coerce")
setores_poa_limpo["densidade_hab_km2"] = pd.to_numeric(setores_poa_limpo["densidade_hab_km2"], errors="coerce")

setores_poa_limpo["is_fcu"] = setores_poa_limpo["CD_TIPO"].astype("string").eq("1")
setores_poa_limpo["CD_FCU"] = setores_poa_limpo["CD_FCU"].astype("string").replace(".", pd.NA)
setores_poa_limpo["is_rural"] = setores_poa_limpo["CD_SIT"].astype("string").isin({"5", "7", "8"})
setores_poa_limpo["geometry"] = gpd.GeoSeries(setores_poa_limpo["geometry"], crs="EPSG:4674")

assert len(setores_poa_limpo) == 2730
assert int(setores_poa_limpo["is_fcu"].sum()) == 323
assert int(setores_poa_limpo["is_rural"].sum()) == 21
assert int((setores_poa_limpo["v0001"] == 0).sum()) == 21
setores_schema = [
    "CD_SETOR", "NM_BAIRRO", "CD_SIT", "CD_TIPO", "CD_FCU", "NM_FCU",
    "v0001", "AREA_KM2", "densidade_hab_km2", "idosos_60_mais", "pct_idosos",
    "V06001", "V06004", "V06006", "is_fcu", "is_rural", "geometry",
]
setores_poa_limpo = setores_poa_limpo.loc[:, setores_schema]
setores_poa_limpo.to_file(DONE / "setores_poa_limpo.gpkg", layer="setores_poa_limpo", driver="GPKG")

# ---------------- CNES ----------------
cnes_ativos = cnes_poa.loc[cnes_poa["CO_MOTIVO_DESAB"].isna()].copy()
tipos_selecionados = {"1", "2", "4", "5", "7", "20", "21", "70", "73"}
cnes_tipos = cnes_ativos.loc[cnes_ativos["TP_UNIDADE"].astype("string").isin(tipos_selecionados)].copy()
natureza_publica = (
    cnes_tipos["CO_NATUREZA_JUR"].astype("string").str.startswith("1", na=False)
    | cnes_tipos["CO_NATUREZA_JUR"].astype("string").eq("2011")
)
cnes_limpo = cnes_tipos.loc[
    cnes_tipos["CO_AMBULATORIAL_SUS"].astype("string").eq("SIM") | natureza_publica
].copy()

cnes_exclusoes_ids = {
    "8604428", "7355106", "7355122", "7533446", "7355114", "2237156",
    "7989954", "8083649", "915742", "4758749", "6957560", "9948007",
    "6957544", "4560191", "804959", "9827153", "9736271",
}
cnes_limpo = cnes_limpo.loc[
    ~cnes_limpo["CO_CNES"].astype("string").isin(cnes_exclusoes_ids)
].copy()

cnes_lat = pd.to_numeric(cnes_limpo["NU_LATITUDE"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
cnes_lon = pd.to_numeric(cnes_limpo["NU_LONGITUDE"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
cnes_limpo = cnes_limpo.loc[cnes_lat.notna() & cnes_lon.notna()].copy()
cnes_lat = cnes_lat.loc[cnes_limpo.index]
cnes_lon = cnes_lon.loc[cnes_limpo.index]

cnes_limpo = gpd.GeoDataFrame(
    cnes_limpo,
    geometry=gpd.points_from_xy(cnes_lon, cnes_lat),
    crs="EPSG:4674",
)

natureza = cnes_limpo["CO_NATUREZA_JUR"].astype("string")
cnes_publico = natureza.str.startswith("1", na=False) | natureza.eq("2011")
cnes_privado_filantropico = natureza.str.startswith("3", na=False)
cnes_limpo["tipo_hospital_sus"] = "outro"
cnes_limpo.loc[
    cnes_limpo["TP_UNIDADE"].astype("string").isin({"5", "7"}) & cnes_publico,
    "tipo_hospital_sus",
] = "publico"
cnes_limpo.loc[
    cnes_limpo["TP_UNIDADE"].astype("string").isin({"5", "7"}) & cnes_privado_filantropico,
    "tipo_hospital_sus",
] = "privado_filantropico_sus"

assert len(cnes_limpo) == 183
assert int(cnes_limpo["tipo_hospital_sus"].eq("publico").sum()) == 8
assert int(cnes_limpo["tipo_hospital_sus"].eq("privado_filantropico_sus").sum()) == 13
cnes_schema = [
    "CO_CNES", "NO_FANTASIA", "TP_UNIDADE", "CO_NATUREZA_JUR",
    "CO_AMBULATORIAL_SUS", "tipo_hospital_sus", "geometry",
]
cnes_limpo = cnes_limpo.loc[:, cnes_schema]
cnes_limpo.to_file(DONE / "cnes_poa_limpo.gpkg", layer="cnes_poa_limpo", driver="GPKG")

# ---------------- Escolas ----------------
escolas_limpo = escolas_poa.loc[
    ~escolas_poa["Restrição de Atendimento"].astype("string").str.contains("PARALISADA", case=False, na=False)
].copy()
escola_lat = pd.to_numeric(escolas_limpo["Latitude"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
escola_lon = pd.to_numeric(escolas_limpo["Longitude"].astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")
escolas_limpo = escolas_limpo.loc[escola_lat.notna() & escola_lon.notna()].copy()
escola_lat = escola_lat.loc[escolas_limpo.index]
escola_lon = escola_lon.loc[escolas_limpo.index]

escolas_limpo = gpd.GeoDataFrame(
    escolas_limpo,
    geometry=gpd.points_from_xy(escola_lon, escola_lat),
    crs="EPSG:4674",
)

cat = escolas_limpo["Categoria Administrativa"].astype("string").str.strip()
conv = escolas_limpo["Conveniada Poder Público"].astype("string").str.strip()
escolas_limpo["rede_score"] = "Privada Particular"
escolas_limpo.loc[cat.eq("Pública"), "rede_score"] = "Pública"
escolas_limpo.loc[cat.eq("Privada") & conv.eq("Sim"), "rede_score"] = "Privada Conveniada"
etapas_escola = escolas_limpo["Etapas e Modalidade de Ensino Oferecidas"].astype("string").str.strip()
escolas_limpo["is_educacao_infantil"] = etapas_escola.str.contains("Educação Infantil", case=False, na=False)
escolas_limpo["is_apenas_educacao_infantil"] = etapas_escola.eq("Educação Infantil")
escolas_limpo["is_educacao_especial"] = escolas_limpo["Restrição de Atendimento"].astype("string").str.contains("DEFICIÊNCIA", case=False, na=False)

assert len(escolas_limpo) == 809
assert int(escolas_limpo["rede_score"].eq("Pública").sum()) == 313
assert int(escolas_limpo["rede_score"].eq("Privada Conveniada").sum()) == 215
assert int(escolas_limpo["rede_score"].eq("Privada Particular").sum()) == 281
assert int(escolas_limpo["Escola"].astype("string").str.strip().eq("EMEF SAO PEDRO").sum()) == 1
assert int(escolas_limpo["is_educacao_especial"].sum()) == 14
escolas_schema = [
    "Código INEP", "Escola", "Categoria Administrativa", "Dependência Administrativa",
    "Conveniada Poder Público", "Porte da Escola",
    "Etapas e Modalidade de Ensino Oferecidas", "rede_score",
    "is_educacao_infantil", "is_apenas_educacao_infantil", "is_educacao_especial",
    "geometry",
]
escolas_limpo = escolas_limpo.loc[:, escolas_schema]
escolas_limpo.to_file(DONE / "escolas_poa_limpo.gpkg", layer="escolas_poa_limpo", driver="GPKG")

# ---------------- Assistência Social ----------------
assistencia_limpo = assistencia_poa.copy()
assistencia_limpo = gpd.GeoDataFrame(assistencia_limpo, geometry=assistencia_limpo.geometry, crs="EPSG:4674")

assert len(assistencia_limpo) == 35
assert assistencia_limpo["categoria"].value_counts().to_dict() == {"CRAS": 23, "CREAS": 9, "Centro POP": 3}
assistencia_schema = ["categoria", "nome", "endereco", "latitude", "longitude", "geometry"]
assistencia_limpo = assistencia_limpo.loc[:, assistencia_schema]
assistencia_limpo.to_file(DONE / "assistencia_poa_limpo.gpkg", layer="assistencia_poa_limpo", driver="GPKG")

# ---------------- FCUs (opcional) ----------------
fcu_poa_limpo = fcu_poa.copy()
fcu_poa_limpo = gpd.GeoDataFrame(fcu_poa_limpo, geometry=fcu_poa_limpo.geometry, crs="EPSG:4674")
assert len(fcu_poa_limpo) == 125
fcu_poa_limpo = fcu_poa_limpo.loc[:, ["cd_fcu", "nm_fcu", "geometry"]]
fcu_poa_limpo.to_file(DONE / "fcu_poa_contornos.gpkg", layer="fcu_poa_contornos", driver="GPKG")

print("Arquivos exportados em:", DONE)
print("Setores:", len(setores_poa_limpo))
print("CNES:", len(cnes_limpo), "(públicos:", int(cnes_limpo["tipo_hospital_sus"].eq("publico").sum()), "privados/filantrópicos:", int(cnes_limpo["tipo_hospital_sus"].eq("privado_filantropico_sus").sum()), ")")
print("Escolas:", len(escolas_limpo))
print("Assistência Social:", len(assistencia_limpo))
print("FCUs:", len(fcu_poa_limpo))

Arquivos exportados em: /Users/aluno-81/Documents/projects/eld_dataset/data/3_done
Setores: 2730
CNES: 183 (públicos: 8 privados/filantrópicos: 13 )
Escolas: 809
Assistência Social: 35
FCUs: 125
